# XGBoost Regressor com GridSearch (mesmo dataset e mesmas features)

Este notebook treina um modelo XGBoost para prever `trip_duration` no dataset NYC Taxi usando as mesmas features de tempo/geoespaciais usadas anteriormente.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.metrics import mean_squared_error

from scipy.stats import randint, uniform, loguniform
from xgboost import XGBRegressor

In [7]:
base_path = Path("..") / "datasets" / "nyc-taxi-trip-duration"
train_path = base_path / "train" / "train.csv"

df = pd.read_csv(train_path)
print(f"Loaded: {train_path} -> {df.shape}")

def add_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df_feat = df_in.copy()

    df_feat["pickup_datetime"] = pd.to_datetime(df_feat["pickup_datetime"], errors="coerce")
    df_feat["pickup_weekday_1_7"] = df_feat["pickup_datetime"].dt.weekday + 1
    df_feat["pickup_hour"] = df_feat["pickup_datetime"].dt.hour

    df_feat["pickup_month"] = df_feat["pickup_datetime"].dt.month
    df_feat["pickup_day"] = df_feat["pickup_datetime"].dt.day
    df_feat["pickup_minute"] = df_feat["pickup_datetime"].dt.minute
    df_feat["is_weekend"] = (df_feat["pickup_datetime"].dt.weekday >= 5).astype(int)

    # Mesmo estado do dia usado no notebook principal
    bins = [0, 6, 12, 18, 24]
    labels = ["Morning", "Afternoon", "Evening", "Night"]
    df_feat["state_of_day"] = pd.cut(
        df_feat["pickup_hour"],
        bins=bins,
        labels=labels,
        include_lowest=True,
        ordered=False,
    ).astype(str)

    df_feat["pickup_hour_sin"] = np.sin(2 * np.pi * df_feat["pickup_hour"] / 24)
    df_feat["pickup_hour_cos"] = np.cos(2 * np.pi * df_feat["pickup_hour"] / 24)

    # Rush hour com mesma ideia (top_k=4, min_rides=500)
    temp = df_feat.dropna(subset=["pickup_hour", "trip_duration"]).copy()
    rush_hours = []
    if not temp.empty:
        hour_stats = temp.groupby("pickup_hour", as_index=False).agg(
            mean_trip_duration=("trip_duration", "mean"),
            ride_count=("trip_duration", "size"),
        )
        hour_stats = hour_stats[hour_stats["ride_count"] >= 500]
        if not hour_stats.empty:
            rush_hours = (
                hour_stats.sort_values("mean_trip_duration", ascending=False)
                .head(4)["pickup_hour"]
                .astype(int)
                .tolist()
            )

    df_feat["is_rush_hour"] = df_feat["pickup_hour"].isin(rush_hours).astype(int)

    return df_feat

df_model = add_features(df)
print("Features added successfully.")

Loaded: ..\datasets\nyc-taxi-trip-duration\train\train.csv -> (1458644, 11)
Features added successfully.


In [8]:
feature_cols = [
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "pickup_weekday_1_7",
    "pickup_hour",
    "pickup_hour_sin",
    "pickup_hour_cos",
    "pickup_month",
    "pickup_day",
    "pickup_minute",
    "is_weekend",
    "is_rush_hour",
    "state_of_day",
]

target_col = "trip_duration"

available_features = [c for c in feature_cols if c in df_model.columns]
missing_features = [c for c in feature_cols if c not in df_model.columns]

print("Using features:", available_features)
if missing_features:
    print("Missing features:", missing_features)

model_df = df_model[available_features + [target_col]].dropna().copy()

X = model_df[available_features]
y = model_df[target_col]

y_log = np.log1p(y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

categorical_features = ["state_of_day"] if "state_of_day" in available_features else []
numeric_features = [c for c in available_features if c not in categorical_features]

# Robust normalization para reduzir efeito de outliers antes do XGBoost
numeric_pipeline = Pipeline(
    steps=[
        ("scaler", RobustScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", numeric_pipeline, numeric_features),
    ],
    remainder="drop",
)

print(f"Train shape: {X_train.shape}, Valid shape: {X_valid.shape}")

Using features: ['passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_weekday_1_7', 'pickup_hour', 'pickup_hour_sin', 'pickup_hour_cos', 'pickup_month', 'pickup_day', 'pickup_minute', 'is_weekend', 'is_rush_hour', 'state_of_day']
Train shape: (1166915, 15), Valid shape: (291729, 15)


In [ ]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", xgb),
    ]
)

# Random Search com 100 combinações.
# Obs.: o XGBoost já otimiza por gradiente boosting internamente;
# aqui estamos fazendo busca aleatória dos hiperparâmetros.
param_distributions = {
    "model__n_estimators": randint(200, 900),
    "model__max_depth": randint(4, 11),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__subsample": uniform(0.7, 0.3),
    "model__colsample_bytree": uniform(0.7, 0.3),
    "model__min_child_weight": randint(1, 10),
    "model__gamma": uniform(0.0, 0.5),
    # L1 / L2 regularization (estilo Elastic Net no XGBoost)
    "model__reg_alpha": loguniform(1e-4, 1.0),
    "model__reg_lambda": loguniform(1e-2, 10.0),
}

grid = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=100,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV RMSE (log-space):", np.sqrt(-grid.best_score_))

Fitting 3 folds for each of 100 candidates, totalling 300 fits


In [ ]:
best_model = grid.best_estimator_

pred_valid_log = best_model.predict(X_valid)
pred_valid = np.expm1(pred_valid_log)

y_valid_real = np.expm1(y_valid)

rmse = np.sqrt(mean_squared_error(y_valid_real, pred_valid))
rmsle = np.sqrt(mean_squared_error(np.log1p(y_valid_real), np.log1p(np.maximum(pred_valid, 0))))

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation RMSLE: {rmsle:.4f}")

results_preview = pd.DataFrame({
    "y_true": y_valid_real[:10],
    "y_pred": pred_valid[:10],
})
results_preview

Validation RMSE: 3198.8969
Validation RMSLE: 0.4023


,y_true,y_pred
67250,1040.0,799.641174
1397036,827.0,749.123474
1021087,614.0,435.484558
951424,867.0,1053.011475
707882,4967.0,5139.857910
1432685,374.0,359.285797
528407,1252.0,513.116943
1217989,148.0,184.434998
1165122,1499.0,1396.116943
1324437,1017.0,724.000000
